In [1]:
# ==================================================
# GS-STRE Phase 1: Generative Seeding (Annotator)
# Jupyter Notebook Version
# ==================================================

import os
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


In [5]:
!pip install openai


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ------------------------------
# 1. Setup
# ------------------------------
# Make sure you set your OpenAI API key before running:
# Linux/Mac:  export OPENAI_API_KEY="sk-xxxx"
# Windows PS: setx OPENAI_API_KEY "sk-xxxx"

with open('C:/Users/User/Documents/master/research/code/code 2/key1.txt') as f:
    key = f.read()
    client = OpenAI(api_key=key)

#INPUT_FILE = "C:/Users/User/Documents/master/research/code/code 2/first_30_samples.tsv" 
INPUT_FILE = "C:/Users/User/Documents/master/research/code/code 2/lotus_with_title_abstract.tsv"   # your enriched TSV
#OUTPUT_FILE = "C:/Users/User/Documents/master/research/code/code 2/lotus_seed_dataset.json"        # final annotated dataset
CHUNK_DIR = "C:/Users/User/Documents/master/research/code/code 2/chunks"                           # where smaller files go
os.makedirs(CHUNK_DIR, exist_ok=True)



In [3]:
# ------------------------------
# 2. Split the input into smaller parts
# ------------------------------
def split_input_file(input_file, rows_per_chunk=5000):
    df = pd.read_csv(input_file, sep="\t")
    total_rows = len(df)
    print(f"📊 Total rows: {total_rows}")

    chunks = []
    for i in range(0, total_rows, rows_per_chunk):
        chunk_df = df.iloc[i:i+rows_per_chunk]
        chunk_file = os.path.join(CHUNK_DIR, f"lotus_chunk_{i//rows_per_chunk+1}.tsv")
        chunk_df.to_csv(chunk_file, sep="\t", index=False)
        chunks.append(chunk_file)
        print(f"✅ Saved {len(chunk_df)} rows to {chunk_file}")

    return chunks

# Run splitting once before annotation
chunks = split_input_file(INPUT_FILE, rows_per_chunk=5000)
print(f"📂 Created {len(chunks)} chunk files in {CHUNK_DIR}/")


📊 Total rows: 122733
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_1.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_2.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_3.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_4.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_6.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_8.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv
✅ Saved 5000 rows to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_10.ts

In [3]:
# ------------------------------
# 2. Prompt Template
# ------------------------------
PROMPT_TEMPLATE = """You are an expert biocurator. Your task is to find the precise textual evidence 
for a known relationship within a scientific abstract.

Known Relationship:
ORGANISM: {organism}
CHEMICAL: {chemical}

Abstract Text:
{abstract}

Instructions:
1. Carefully read the abstract to locate the single best sentence that explicitly states or strongly implies 
   that the given ORGANISM produces or was the source of the given CHEMICAL.
2. From that sentence, extract the exact textual mentions (spans) for the organism and the chemical 
   as they appear in the text. The spans must be a perfect substring of the sentence.
3. If no sentence in the abstract provides clear evidence for this specific relationship, you MUST return "null".

Output Format (JSON): 
{{
  "supporting_sentence": "...",
  "organism_span": "...",
  "chemical_span": "..."
}}
"""



In [4]:
# ------------------------------
# 3. Query GPT for a single relation
# ------------------------------
import re

def query_llm(organism, chemical, abstract):
    if pd.isna(abstract) or not str(abstract).strip():
        return None

    prompt = PROMPT_TEMPLATE.format(
        organism=organism,
        chemical=chemical,
        abstract=abstract
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        output = response.choices[0].message.content.strip()

        if "null" in output.lower():
            return None

        # Try direct JSON parsing
        try:
            return json.loads(output)
        except json.JSONDecodeError:
            # Attempt to extract JSON block with regex
            match = re.search(r"\{.*\}", output, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(0))
                except Exception:
                    pass
            print(f"⚠️ Invalid JSON response for {organism}-{chemical}: {output[:200]}...")
            return None

    except Exception as e:
        print(f"⚠️ API error for {organism}-{chemical}: {e}")
        return None



In [5]:
# ------------------------------
# 5. Annotate one chunk at a time
# ------------------------------
def process_chunk(chunk_file, output_file):
    df = pd.read_csv(chunk_file, sep="\t")
    seed_data = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {chunk_file}"):
        organism = row["organism_name"]
        chemical = row["structure_nameTraditional"]
        abstract = row["abstract"]

        result = query_llm(organism, chemical, abstract)
        if result:
            seed_data.append({
                "pubmed_id": row["reference_pubmed_id"],
                "title": row["title"],
                "abstract": abstract,
                "organism_name": organism,
                "chemical_name": chemical,
                "supporting_sentence": result.get("supporting_sentence", ""),
                "organism_span": result.get("organism_span", ""),
                "chemical_span": result.get("chemical_span", "")
            })

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(seed_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved {len(seed_data)} annotated relations to {output_file}")


In [7]:

# Example: process first chunk
chunk_to_run = chunks[0]
output_for_chunk = chunk_to_run.replace(".tsv", "_seed.json")
process_chunk(chunk_to_run, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_1.tsv:  60%|██████    | 3015/5000 [5:03:21<1:40:29,  3.04s/it]     

⚠️ Invalid JSON response for Moringa oleifera-Ethyl 4-(rhamnosyloxy)benzylcarbamate: ```json
{
  "supporting_sentence": "Two new compounds, O-[2'-hydroxy-3'-(2"-heptenyloxy)]-propyl undecanoate (1) and O-ethyl-4-[(alpha-L-rhamnosyloxy)-benzyl] carbamate (2) along with the known substa...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_1.tsv: 100%|██████████| 5000/5000 [6:28:53<00:00,  4.67s/it]  

✅ Saved 3279 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_1_seed.json


In [8]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_2.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_2.tsv: 100%|██████████| 5000/5000 [3:39:13<00:00,  2.63s/it]   

✅ Saved 3490 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_2_seed.json


In [6]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_3.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_3.tsv: 100%|██████████| 5000/5000 [3:38:55<00:00,  2.63s/it]    


✅ Saved 3616 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_3_seed.json


In [7]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_4.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_4.tsv: 100%|██████████| 5000/5000 [6:28:37<00:00,  4.66s/it]       

✅ Saved 3550 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_4_seed.json


In [6]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_5.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv:  31%|███       | 1530/5000 [1:06:28<2:09:33,  2.24s/it]

⚠️ Invalid JSON response for Euphorbia quinquecostata-Xanthoxylin: ```json
{
  "supporting_sentence": "Also isolated from this extract were 10 constituents inactive in this bioassay, namely, 2,2'-dihydroxy-4,6-dimethoxy-3-methylacetophenone (5), a new structure, and ...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv:  31%|███       | 1534/5000 [1:06:45<3:31:49,  3.67s/it]

⚠️ Invalid JSON response for Euphorbia quinquecostata-Beta-Sitosterol: ```json
{
  "supporting_sentence": "Also isolated from this extract were 10 constituents inactive in this bioassay, namely, 2,2'-dihydroxy-4,6-dimethoxy-3-methylacetophenone (5), a new structure, and ...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv:  31%|███       | 1539/5000 [1:07:16<5:54:22,  6.14s/it]

...

⚠️ Invalid JSON response for Plocama calabrica-Paederosidic acid: ```json
{
  "supporting_sentence": "From the aerial parts of Putoria calabrica, two new flavonol triglycosides were isolated and their structures were elucidated as quercetin-3-O-[alpha-L-rhamnopyrano...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv:  89%|████████▉ | 4441/5000 [3:26:23<6:15:26, 40.30s/it]

⚠️ API error for Inula japonica-Asperilin: Connection error.


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5.tsv: 100%|██████████| 5000/5000 [3:50:20<00:00,  2.76s/it]  

✅ Saved 3647 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_5_seed.json


In [7]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_6.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_6.tsv: 100%|██████████| 5000/5000 [4:23:59<00:00,  3.17s/it]   

✅ Saved 4060 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_6_seed.json


In [7]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_7.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv:  80%|████████  | 4005/5000 [2:53:22<1:01:55,  3.73s/it]

⚠️ Invalid JSON response for Carapichea ipecacuanha-Neocephaeline: ```json
{
  "supporting_sentence": "From the dried roots of Cephaelis acuminata, five ipecac alkaloids, neocephaeline, 7'-O-demethylcephaeline, 10-O-demethylcephaeline, 2'-N-(1"-deoxy-1"-beta-D-fructo...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv:  80%|████████  | 4006/5000 [2:53:27<1:06:13,  4.00s/it]

⚠️ Invalid JSON response for Carapichea ipecacuanha-Psychotrine: ```json
{
  "supporting_sentence": "From the dried roots of Cephaelis acuminata, five ipecac alkaloids, neocephaeline, 7'-O-demethylcephaeline, 10-O-demethylcephaeline, 2'-N-(1"-deoxy-1"-beta-D-fructo...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv:  80%|████████  | 4007/5000 [2:53:31<1:04:07,  3.87s/it]

...

⚠️ Invalid JSON response for Carapichea ipecacuanha-Emetine: ```json
{
  "supporting_sentence": "From the dried roots of Cephaelis acuminata, five ipecac alkaloids, neocephaeline, 7'-O-demethylcephaeline, 10-O-demethylcephaeline, 2'-N-(1"-deoxy-1"-beta-D-fructo...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv:  80%|████████  | 4011/5000 [2:53:46<1:04:46,  3.93s/it]

⚠️ Invalid JSON response for Carapichea ipecacuanha-10-Demethylcephaeline: ```json
{
  "supporting_sentence": "From the dried roots of Cephaelis acuminata, five ipecac alkaloids, neocephaeline, 7'-O-demethylcephaeline, 10-O-demethylcephaeline, 2'-N-(1"-deoxy-1"-beta-D-fructo...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7.tsv: 100%|██████████| 5000/5000 [4:52:22<00:00,  3.51s/it]     

✅ Saved 4056 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_7_seed.json


In [8]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_8.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_8.tsv: 100%|██████████| 5000/5000 [6:43:47<00:00,  4.85s/it]      


✅ Saved 4196 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_8_seed.json


In [6]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_9.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv:  65%|██████▌   | 3259/5000 [2:11:19<1:45:35,  3.64s/it] 

⚠️ Invalid JSON response for Juglans nigra-3,3',4',5,7-Pentahydroxyflavanone: ```json
{
  "supporting_sentence": "From the stem-bark of Juglans mandshurica, two new naphthalenyl glucopyranosides, 1,4,8-trihydroxynaphthalene 1-O-[alpha-L-arabinofuranosyl-(1-->6)-beta-D-glucopyra...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv:  65%|██████▌   | 3260/5000 [2:11:25<2:10:33,  4.50s/it]

⚠️ Invalid JSON response for Juglans mandshurica-alpha-Hydrojuglone 4-glucoside: ```json
{
  "supporting_sentence": "From the stem-bark of Juglans mandshurica, two new naphthalenyl glucopyranosides, 1,4,8-trihydroxynaphthalene 1-O-[alpha-L-arabinofuranosyl-(1-->6)-beta-D-glucopyra...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv:  65%|██████▌   | 3261/5000 [2:11:31<2:24:10,  4.97s/it]

...

⚠️ Invalid JSON response for Juglans mandshurica-Myricitrin: ```json
{
  "supporting_sentence": "From the stem-bark of Juglans mandshurica, two new naphthalenyl glucopyranosides, 1,4,8-trihydroxynaphthalene 1-O-[alpha-L-arabinofuranosyl-(1-->6)-beta-D-glucopyra...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv:  66%|██████▌   | 3291/5000 [2:13:11<1:00:53,  2.14s/it]

⚠️ Invalid JSON response for Searsia pyroides-rhuschalcone II: ```json
{
  "supporting_sentence": "These new flavonoids belong to a rare bichalcone class and have been identified as 2',4',4' ',2' ",4' "-pentahydroxy-4-O-5' "-bichalcone (rhuschalcone II, 2)...",
 ...


Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9.tsv: 100%|██████████| 5000/5000 [3:26:14<00:00,  2.47s/it]  

✅ Saved 4024 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_9_seed.json


In [6]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_10.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_10.tsv: 100%|██████████| 5000/5000 [4:33:55<00:00,  3.29s/it]  


✅ Saved 4262 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_10_seed.json


In [6]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_11.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_11.tsv: 100%|██████████| 5000/5000 [4:30:20<00:00,  3.24s/it]  

✅ Saved 4200 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_11_seed.json


In [7]:
chunk_file = os.path.join(CHUNK_DIR, "lotus_chunk_12.tsv")
output_for_chunk = chunk_file.replace(".tsv", "_seed.json")

process_chunk(chunk_file, output_for_chunk)

Processing C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_12.tsv: 100%|██████████| 5000/5000 [5:14:24<00:00,  3.77s/it]   


✅ Saved 4290 annotated relations to C:/Users/User/Documents/master/research/code/code 2/chunks\lotus_chunk_12_seed.json
